# ESM Variant-Effect Tools — Google Colab

Run the [**ESM-variant-effect-tools**](https://github.com/kouroshSA/ESM-variant-effect-tools) 
toolkit directly in your browser — no local install required. These tools predict and analyze 
the effects of protein mutations using **ESM (Evolutionary Scale Modeling)** protein language models.

**Before you start:** enable a GPU. Go to **Runtime → Change runtime type → Hardware accelerator → GPU**, 
then run the cells below in order.

Steps in this notebook:
1. Check the GPU and install the toolkit
2. Predict per-substitution mutation effects (fast demo over a small position window)
3. Group variant analysis — cumulative score, PCA, and nearest-neighbors
4. Visualize the predicted mutation effects
5. Run the tools on **your own** sequences

## 0. Check the GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — enable one via Runtime > Change runtime type > GPU for reasonable speed.')

## 1. Install the toolkit

This clones the repository and installs the few extra dependencies that Colab does not already ship 
(PyTorch, pandas, numpy, scikit-learn, matplotlib and seaborn come preinstalled).

In [ ]:
import os
if not os.path.isdir('ESM-variant-effect-tools'):
    !git clone --depth 1 https://github.com/kouroshSA/ESM-variant-effect-tools.git
%cd ESM-variant-effect-tools
!pip install -q fair-esm biopython annoy logomaker
print('\nRepository contents:')
!ls -1

## 2. Predict mutation effects (fast demo)

The predictor enumerates single amino-acid substitutions in each input sequence and scores them with an 
**ESM-1v** model (the *delta log-probability* relative to the wild-type residue). The pretrained weights 
(~2.5 GB) download automatically on first run.

To keep this demo quick, we restrict scoring to a small window of positions with `--ranges` 
(here `24-40`). **Remove the `--ranges` flag to score every position** in the protein (more informative, 
but several minutes per sequence). You can also target individual residues with, e.g., `--positions 33 66`.

In [ ]:
# Window of positions to score for the fast demo (edit or remove --ranges for a full scan):
DEMO_RANGE = '24-40'

# Reference WT predictions (used by the group analyzer in step 3):
!python ESM_predict_mutation_effects.py WT_LDHA.fasta WT_LDHA_predictions.csv --cuda --ranges {DEMO_RANGE}

# Variant predictions (used by the visualizer in step 4):
!python ESM_predict_mutation_effects.py LDH_variants.fasta LDH_variants_predictions.csv --cuda --ranges {DEMO_RANGE}

## 3. Group variant analysis (cumulative score + PCA + Annoy)

Aligns each variant to the wild type, computes a cumulative detrimental score, and runs PCA plus an 
Annoy nearest-neighbor search over the per-variant mutation-effect vectors.

> **Note:** because the fast demo only scored the `24-40` window, substitutions outside that window are 
treated as neutral (score 0), so the cumulative scores here reflect just that window. Re-run step 2 
without `--ranges` for whole-protein scores.

In [ ]:
!python calculate_variant_scores_with_proxy_PCA-annoy.py \
    WT_LDHA.fasta LDH_variants.fasta WT_LDHA_predictions.csv cumulative_scores.csv --perform_pca

import pandas as pd
print('\nCumulative scores:')
display(pd.read_csv('cumulative_scores.csv'))

## 4. Visualize the predicted mutation effects

Generates, for each sequence, a heatmap, a per-position boxplot, an average-effect bar plot by amino acid, 
and a sequence logo. It aligns each variant to the WT, so insertions/deletions are handled correctly. 
The figures cover whatever positions were scored in step 2 (the `24-40` window for the fast demo).

In [ ]:
!python visualize_mutation_effects_indel_adjust_logo2.py \
    LDH_variants_predictions.csv viz_out \
    --wt_sequence WT_LDHA.fasta --variant_sequences LDH_variants.fasta

# Display the generated figures inline:
import glob
from IPython.display import Image, display
images = sorted(glob.glob('viz_out/*.png'))
print(f'{len(images)} figures generated.')
for img in images:
    print(img)
    display(Image(filename=img))

## 5. Use your own sequences

Upload your own FASTA file and run the predictor on it. Run the cell, choose a `.fasta` file, then it will 
score substitutions and show the first few rows of the output.

The cell uses the same `--ranges` window as the demo so it stays fast; **remove `--ranges {DEMO_RANGE}` 
for a full scan.** For the group analysis and visualization steps, provide your own wild-type and variant 
FASTA files the same way and reuse the commands from steps 3 and 4 with your filenames.

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose a .fasta file

import pandas as pd
for fname in uploaded:
    out = fname.rsplit('.', 1)[0] + '_predictions.csv'
    !python ESM_predict_mutation_effects.py "{fname}" "{out}" --cuda --ranges {DEMO_RANGE}
    print(f'\nPredictions written to {out}:')
    display(pd.read_csv(out).head())

---
Original code © Kourosh Salehi-Ashtiani, MIT License. Built on the ESM models / `fair-esm` by Meta AI / FAIR 
(MIT License). Please cite the ESM publications listed in the 
[repository README](https://github.com/kouroshSA/ESM-variant-effect-tools#acknowledgments--citation) 
if you use these tools in academic work.